# 04 — Configuration, and how to measure a change instead of guessing

`Config` holds every number that governs the cascade, and each default carries in
its own description the measurement that fixed it. Changing them is legitimate.
Changing them **without measuring** is what this notebook exists to make
difficult (CLAUDE.md §8).

There is a second rule in that section, and it is worth more than any specific
number:

> **An isolated measurement has already lied here.** Turning off Vision's
> language correction improved anchor preservation across 60 documents (+4) and
> worsened it across the whole cascade (−227), because the worse text failed the
> gate and fell to worse engines. What decides is the cascade's behaviour, not
> one engine's output.

So the second half of this notebook measures one knob **twice** — once at the
engine and once at the cascade — and the two answers disagree.

In [ ]:
import logging

logging.disable(logging.INFO)

from autosxtract import Config, resources

config = Config()
print("this machine:", resources.describe())
print()
print("fields:", len(Config.model_fields))

## Reading the configuration as documentation

The descriptions are not filler: they are where the measurement lives, next to
the number it justifies. Printing them is the fastest way to learn what the
library actually cares about — and it is the version installed here rather than
whatever a web page says.

In [ ]:
HEADLINE = [
    "use_native", "dpi", "grayscale",
    "min_useful_words", "min_chars_per_page", "min_score", "native_accept_score",
    "coverage_gate", "consensus_gate", "agreement_gate", "min_agreement",
    "expensive_step_vetoes", "replacement_gate", "veto_engine",
    "layers", "layer2", "per_page_routing", "rebuild_prose",
]

for name in HEADLINE:
    field = Config.model_fields[name]
    sentence = (field.description or "").split(". ")[0].rstrip(".")
    print(f"{name:<24} = {getattr(config, name)!r:<14} {sentence}.")

Two of those deserve to be read twice.

`min_useful_words = 12` is the floor of alphabetic words **outside the stamp**,
derived from an audit of 1,339 documents in which 403 had text that looked fine
and was only the digital signature banner. It is the same number the consensus
gate votes against, which is not a coincidence: one criterion, one threshold.

`veto_engine = "tesseract"` names the *witness*, and the witness never
transcribes. It must be of a **different architecture** from the engines already
run — a second engine of the same family is not independent evidence, and the
agreement veto would stop meaning what it says. That is why it points at
Tesseract and not at a second PP-OCR.

## What is deliberately not in there

`Config` has **no host, port, URL or credential field**, and that is an
architectural invariant rather than an oversight (CLAUDE.md §1). All networking
lives in the constructor of a step somebody wrote by hand, so a remote step
nobody declared cannot exist. The reason is measured: the previous version of
this pipeline reached an OCR engine over a reverse SSH tunnel, and that worker
going down **silently degraded the text** — 488 documents re-extracted down the
worse path, 19.5 minutes instead of 4.9, 28,239 characters lost, and nobody
noticed until somebody checked.

In [ ]:
network_words = ("host", "port", "url", "endpoint", "token", "password", "credential")
suspects = [n for n in Config.model_fields if any(w in n.lower() for w in network_words)]
print("fields that point at a network:", suspects or "none")

# The configuration is frozen and forbids unknown keys, so a typo is a loud
# failure at construction rather than a setting that silently does nothing.
for attempt in ("Config(dpiz=200)", "Config(dpi=10)"):
    try:
        eval(attempt)
    except Exception as exc:
        print(f"\n{attempt} ->", type(exc).__name__)
        print("   ", str(exc).splitlines()[1:3])

## Parallelism resolves in a *method*, not in a field

The three parallelism fields accept `None`, meaning "decide from this machine",
and that is the default because the same library runs on a 2-core laptop and a
72-core server, and one fixed number serves both badly.

They resolve in methods rather than in computed fields for a specific reason:
**the machine that resolves may not be the one that serialised the
configuration.** A preset stored in YAML and used in two environments has to give
different answers in each.

The measured curve behind the ceiling:

```
threads   72 cores     2 cores
   1       1.36 pg/s   1.44 pg/s
   2       1.74        1.68     <- plateau
   4       1.99        1.58
   8       2.18        1.54     <- worse than 2 threads
  16       2.27        1.71
```

An explicit number is **obeyed** — whoever knows their hardware decides. What it
is not, is a promise.

In [ ]:
print("cores seen here         :", resources.cores())
print("automatic per-document  :", config.pages_in_flight())
print("automatic per-batch     :", config.documents_in_flight())
print("batch (documents, pages):", config.batch_concurrency())
print()

# The aggregate cap cuts the PAGES, never the documents: cutting documents raises
# total time predictably, cutting pages per document costs almost nothing.
greedy = Config(document_parallelism=8, page_parallelism=8, concurrency_cap=16)
print("asked for  : 8 documents x 8 pages = 64 pages in flight")
print("capped to  :", greedy.batch_concurrency(), "-> the pages gave way, not the documents")

`os.cpu_count()` lies inside a container — it reports the host, not the quota —
so `resources.cores()` crosses CPU affinity, cgroup v1 and v2 and `cpu_count` and
keeps the smallest. None of the three alone covers `taskset` **and** `--cpus`.

## Now the measurement

The knob under test is `dpi`. Its default of 150 carries this note: *"150 is the
minimum that preserves numeric anchors: at 100 DPI preservation falls to 85.5%,
losing dates and tax numbers in 35 of 60 documents."* That was measured on a real
archive of real scans; the synthetic pages below are crisp renders, so **do not
read the numbers here as a verdict on the default** — read the *method*.

The method needs three things: a corpus, a ground truth, and two levels of
measurement. Numeric anchors — case numbers, protocols, dates, a chassis number
— are the ground truth, because they are the tokens whose corruption no text
metric detects: `9XXYZ3ZE...` scores exactly like `9XXYZ32E...`.

In [ ]:
import statistics
import time

import pymupdf

from autosxtract.quality.anchors import anchors

SOURCES = {
    "certidao": (
        "CERTIDAO DE OBJETO E PE\n\nCertifico que nos autos do processo "
        "0001234-56.2020.8.12.0001, protocolo 882167, tramita acao de execucao "
        "proposta pelo exequente em face do executado, tendo sido a diligencia "
        "cumprida em 17/03/2005 conforme mandado expedido por esta vara civel. "
        "Nada mais havendo, encerro a presente certidao."
    ),
    "mandado": (
        "MANDADO DE INTIMACAO\n\nFica a parte executada intimada nos autos do "
        "processo 0009876-54.2019.8.12.0002, protocolo 774521, para "
        "manifestar-se no prazo legal sob pena de preclusao, referente ao "
        "veiculo de chassi 9XXYZ32E41A099887, conforme decisao proferida em "
        "05/11/2018 pelo juiz de direito desta vara."
    ),
    "peticao": (
        "PETICAO\n\nO requerente, nos autos do processo "
        "0004321-98.2021.8.12.0003, protocolo 665439, vem respeitosamente "
        "requerer a juntada dos documentos anexos e a designacao de nova "
        "diligencia, uma vez que a anterior restou infrutifera em 22/08/2020, "
        "conforme certidao do oficial de justica."
    ),
}


def scanned(text: str, dpi: int = 170, fontsize: int = 9) -> bytes:
    """Ink, no text layer — so every character below came through an engine."""
    doc = pymupdf.open()
    doc.new_page().insert_textbox(pymupdf.Rect(50, 50, 550, 400), text, fontsize=fontsize)
    digital = doc.tobytes()
    doc.close()
    source = pymupdf.open("pdf", digital)
    out = pymupdf.open()
    pixmap = source[0].get_pixmap(dpi=dpi, colorspace=pymupdf.csGRAY)
    sheet = out.new_page(width=source[0].rect.width, height=source[0].rect.height)
    sheet.insert_image(sheet.rect, stream=pixmap.tobytes("png"))
    data = out.tobytes()
    out.close()
    source.close()
    return data


archive = {name: scanned(text) for name, text in SOURCES.items()}
truth = {name: anchors(text) for name, text in SOURCES.items()}
total_anchors = sum(len(a) for a in truth.values())
print(f"{len(archive)} synthetic documents, {total_anchors} anchors to preserve")

### Level 1 — the engine, in isolation

This is the measurement that is easy to run and easy to believe: hand the engine
the pixels, count what comes back. No gates, no layers, no contest.

In [ ]:
from autosxtract.engines import available, get
from autosxtract.pdf.render import render
from autosxtract.quality.stamp import useful_words

LEVELS = (72, 150)
ready = [info.name for info in available()]
engine = get(ready[0]) if ready else None

if engine is None:
    print("no OCR engine on this machine — the measurement below cannot run here.")
    print("The method is the lesson; `scripts/compare_engines.py` runs it on your archive.")
else:
    print(f"engine: {engine.name}\n")
    print(f"  {'dpi':>5} {'ms/page':>9} {'useful words':>13} {'anchors kept':>13}")
    for dpi in LEVELS:
        times, words, kept = [], 0, 0
        for name, pdf in archive.items():
            images = render(pdf, dpi=dpi, max_pages=1)
            t0 = time.perf_counter()
            transcription = engine.transcribe(images, parallelism=1)
            times.append((time.perf_counter() - t0) * 1000 / len(images))
            words += useful_words(transcription.text)
            kept += len(truth[name] & anchors(transcription.text))
        print(f"  {dpi:>5} {statistics.median(times):>9.0f} {words:>13} "
              f"{kept:>9}/{total_anchors}")

### Level 2 — the cascade, which is what actually gets persisted

Same documents, same knob, but now measured on the text the library **keeps**:
which step won, how many characters were persisted, and how many of the ground
truth anchors survived into the result.

In [ ]:
from autosxtract import Cascade

if engine is None:
    print("skipped — no engine (see above)")
else:
    print(f"  {'dpi':>5} {'total s':>9} {'chars':>8} {'anchors kept':>13}   steps")
    for dpi in LEVELS:
        cascade = Cascade(Config(dpi=dpi))
        chars, kept, steps = 0, 0, {}
        t0 = time.perf_counter()
        for name, pdf in archive.items():
            result = cascade.extract(pdf, name)
            steps[result.step] = steps.get(result.step, 0) + 1
            chars += len(result.text)
            kept += len(truth[name] & anchors(result.text))
        elapsed = time.perf_counter() - t0
        print(f"  {dpi:>5} {elapsed:>9.1f} {chars:>8} {kept:>9}/{total_anchors}   {steps}")

### Read the two tables against each other

On a machine with PP-OCRv6 the two levels typically disagree on this corpus: the
engine loses anchors at the higher resolution, and the **cascade loses none at
either**. An operator who had only run level 1 would have lowered the default and
believed they had improved the pipeline; the text the library persists would not
have changed at all.

The disagreement is not noise, and it has a name. Between the engine's output and
the cascade's result sit the containment layers — Layer 2 hands the damaged lines
back to the engine to be re-read. The cell below removes exactly that and
reproduces the engine-level loss *inside* the cascade, which is how you confirm a
mechanism instead of asserting one.

In [ ]:
if engine is None:
    print("skipped — no engine")
else:
    print(f"  {'layers':>7} {'dpi':>5} {'chars':>8} {'anchors kept':>13}")
    for layers in (True, False):
        for dpi in LEVELS:
            cascade = Cascade(Config(dpi=dpi, layers=layers))
            chars, kept = 0, 0
            for name, pdf in archive.items():
                result = cascade.extract(pdf, name)
                chars += len(result.text)
                kept += len(truth[name] & anchors(result.text))
            print(f"  {str(layers):>7} {dpi:>5} {chars:>8} {kept:>9}/{total_anchors}")

That is the whole methodological point, and it generalises past `dpi`: **a knob
is only as good as the pipeline around it.** The layers were measured to be worth
entity recall 0.902 → 0.921 with latency *falling* from 298 to 236 ms, and part of
what they buy is exactly this — insensitivity to a setting somebody would
otherwise have spent a week tuning.

`scripts/compare_engines.py` is the same procedure over a real archive, and it
prints both levels side by side for the same reason. Its own docstring carries the
Vision story that motivated it.

## A gate is a setting too

Turning a gate off is a configuration change and deserves the same treatment. The
coverage gate refuses flawless native text when a large image sits in a region
with no text — the filing that embeds an official letter as an image, where the
native layer is perfect and the attachment is never read.

In [ ]:
def filing_with_attachment() -> bytes:
    doc = pymupdf.open()
    page = doc.new_page()
    page.insert_textbox(
        pymupdf.Rect(50, 50, 550, 380),
        "Peticao inicial nos autos do processo 0001234-56.2020.8.12.0001, em "
        "tramite perante a vara civel, em que o requerente pede a citacao do "
        "requerido na forma da decisao anterior, bem como a juntada dos "
        "documentos que seguem em anexo neste mesmo arquivo e cujo teor "
        "integra o pedido para todos os efeitos de direito. O exequente "
        "esclarece que a diligencia anterior restou infrutifera e que o "
        "oficial de justica certificou nos autos a impossibilidade de "
        "cumprimento.",
        fontsize=11,
    )
    pix = pymupdf.Pixmap(pymupdf.csGRAY, pymupdf.IRect(0, 0, 400, 300))
    pix.set_rect(pix.irect, (210,))
    for y in range(20, 280, 24):
        pix.set_rect(pymupdf.IRect(20, y, 380, y + 6), (40,))
    page.insert_image(pymupdf.Rect(60, 400, 540, 740), stream=pix.tobytes("png"))
    data = doc.tobytes()
    doc.close()
    return data


mixed = filing_with_attachment()
for gate in (True, False):
    result = Cascade(Config(coverage_gate=gate)).extract(mixed, "mixed.pdf")
    print(f"coverage_gate={str(gate):<6} step={result.step:<10} chars={len(result.text):<6} "
          f"ms={result.ms:>7.1f}  {result.provenance}")

With the gate off the cascade stops at the native step: faster, flawless text,
and the attachment never looked at. Nothing in the result says anything is
missing — which is precisely why the gate exists and why the *provenance* line is
the field to compare, not the character count.

## The rule

Every threshold in `config.py` carries the measurement that fixed it, in the
description you printed at the top of this notebook. If you change one, the
change is not finished until the comment says what you measured, over what, and
at **which level**. An engine-level number that has not been checked against the
cascade is a hypothesis, not a result.

Next: **05 — writing your own engine**, where the extension point stops being a
promise.